In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 18.9 MB/s eta 0:00:00


In [4]:
import optuna
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer

In [5]:
df = load_breast_cancer(as_frame=True).frame
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(df.drop('target',axis=1),df['target'],test_size=0.2,random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit(X_test)

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# define the objective function
def objective(trail):
    n_estimators = trail.suggest_int('n_estimators',100,500,10)
    max_depth = trail.suggest_int('max_depth',1,10,1)

    model = RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth)
    score = cross_val_score(model,X_train_scaled,y_train,cv=5).mean()

    return score

In [8]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-05-20 05:44:37,035] A new study created in memory with name: no-name-7f71a4ad-16b2-45e8-a775-46a33f053a9b
/tmp/ipykernel_6799/2117647821.py:6: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  n_estimators = trail.suggest_int('n_estimators',100,500,10)
/tmp/ipykernel_6799/2117647821.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.

In [9]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.9648351648351647
Best hyperparameters: {'n_estimators': 250, 'max_depth': 7}


In [10]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.96


In [11]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [12]:
plot_optimization_history(study).show()

In [13]:
plot_parallel_coordinate(study).show()

In [14]:
plot_slice(study).show()

In [15]:
plot_param_importances(study).show()

In [16]:
plot_contour(study).show()

In [8]:
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier
from sklearn.svm import SVC

In [9]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [10]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-05-20 06:11:35,075] A new study created in memory with name: no-name-0cfa0bf6-c073-4b11-907a-f1e7b710d535
[I 2026-05-20 06:11:35,136] Trial 0 finished with value: 0.6285726734053677 and parameters: {'classifier': 'SVM', 'C': 0.6698598450511323, 'kernel': 'sigmoid', 'gamma': 'auto'}. Best is trial 0 with value: 0.6285726734053677.
[I 2026-05-20 06:11:35,194] Trial 1 finished with value: 0.6285726734053677 and parameters: {'classifier': 'SVM', 'C': 0.36732142803895906, 'kernel': 'sigmoid', 'gamma': 'auto'}. Best is trial 0 with value: 0.6285726734053677.
[I 2026-05-20 06:11:36,247] Trial 2 finished with value: 0.9450302079702567 and parameters: {'classifier': 'RandomForest', 'n_estimators': 99, 'max_depth': 9, 'min_samples_split': 6, 'min_samples_leaf': 2, 'bootstrap': True}. Best is trial 2 with value: 0.9450302079702567.
[I 2026-05-20 06:11:42,159] Trial 3 finished with value: 0.9582461949575927 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 146, 'learning_r

In [11]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'GradientBoosting', 'n_estimators': 166, 'learning_rate': 0.06015407899366217, 'max_depth': 17, 'min_samples_split': 7, 'min_samples_leaf': 5}
Best trial accuracy: 0.9670326478447775


In [17]:
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier(n_estimators = 166, learning_rate= 0.06015407899366217, max_depth= 17, min_samples_split= 7, min_samples_leaf = 5)

model.fit(X_train,y_train)
y_pred = model.predict(X_test)

In [19]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.93      0.95        43
           1       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [27]:
import numpy as np

# Create a new data point (replace with actual values if available)
# It should have 30 features, similar to X_test
new_point_data = np.random.rand(1, X_test.shape[1]) * 100 # Example: random data
new_point = pd.DataFrame(new_point_data, columns=X_test.columns)

# Scale the new data point using the *fitted* scaler
# Note: The scaler was fitted on X_train. X_test_scaled was incorrectly fitted on X_test.
# We should use the scaler instance that was fitted on X_train, which is 'scaler'.
scaled_new_point = scaler.transform(new_point)

# Make a prediction for the new data point
prediction = model.predict(scaled_new_point)

print(f'New data point prediction: {prediction[0]}')

New data point prediction: 0


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
